# 04 - Recomendador basado en contenido

Este notebook implementa una primera version de un sistema de recomendacion basado en contenido usando los generos de las peliculas. El objetivo es construir una base sencilla, clara y facil de explicar antes de incorporar tecnicas mas avanzadas.

## 1. Idea general del enfoque

Un recomendador basado en contenido busca peliculas parecidas a una pelicula de referencia a partir de sus caracteristicas. En este caso, la caracteristica principal seran los generos.

Usar generos como primera aproximacion tiene sentido porque es una informacion facil de interpretar y ya esta disponible en el dataset. Si dos peliculas comparten muchos generos, es razonable pensar que pueden resultar parecidas para un usuario.

La similitud coseno mide hasta que punto dos peliculas se parecen comparando sus vectores de caracteristicas. Si dos peliculas tienen generos muy similares, su similitud sera cercana a 1. Si comparten pocos o ningun genero, la similitud sera mas baja.

## 2. Importacion de librerias

In [37]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

base_dir = Path('..').resolve()
sys.path.append(str(base_dir))

from src.recommender import (
    build_genre_feature_matrix,
    calculate_cosine_similarity,
    find_movie_index,
    get_genre_columns,
    recommend_movies_by_genres,
)

## 3. Carga de datos

In [38]:
data_path = base_dir / 'data' / 'processed' / 'movies_clean.csv'

movies_clean = pd.read_csv(data_path)
movies_clean.head()

,movieId,title,genres,year,title_clean,rating_mean,rating_count,Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,3.897438,68997,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,3.275758,28904,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,3.139447,13134,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,2.845331,2806,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,3.059602,13154,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 4. Identificacion de columnas de genero

Primero detectamos que columnas del dataset corresponden a generos codificados con one-hot encoding.

In [39]:
genre_columns = get_genre_columns(movies_clean)
print('Columnas de genero:', genre_columns)
print('Numero de generos:', len(genre_columns))

Columnas de genero: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
Numero de generos: 19


## 5. Matriz de caracteristicas

A partir de los generos construimos una matriz en la que cada fila representa una pelicula y cada columna indica si pertenece o no a un genero.

In [40]:
feature_matrix = build_genre_feature_matrix(movies_clean, genre_columns)
print('Forma de la matriz de caracteristicas:', feature_matrix.shape)
feature_matrix.head()

Forma de la matriz de caracteristicas: (87585, 19)


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 6. Similitud coseno entre peliculas

La funcion `calculate_cosine_similarity` permite obtener una matriz de similitud coseno entre peliculas. Como el dataset completo es grande, aqui mostramos el calculo sobre un peque?o subconjunto para ilustrar el procedimiento de forma clara.

In [41]:
sample_similarity_matrix = calculate_cosine_similarity(feature_matrix.head(5))
print('Forma de la matriz de similitud de ejemplo:', sample_similarity_matrix.shape)
print(sample_similarity_matrix)

Forma de la matriz de similitud de ejemplo: (5, 5)
[[1.         0.77459667 0.31622777 0.25819889 0.4472136 ]
 [0.77459667 1.         0.         0.         0.        ]
 [0.31622777 0.         1.         0.81649658 0.70710678]
 [0.25819889 0.         0.81649658 1.         0.57735027]
 [0.4472136  0.         0.70710678 0.57735027 1.        ]]


## 7. Busqueda flexible de peliculas

Antes de recomendar, conviene comprobar que la funcion de busqueda encuentra bien la pelicula de entrada incluso cuando el usuario no escribe el titulo completo.

In [42]:
movie_index = find_movie_index(movies_clean, 'Toy Story')
print('Indice encontrado:', movie_index)

Indice encontrado: 0


## 8. Mejora sencilla: minimo de valoraciones

La similitud coseno sigue siendo el criterio principal, pero conviene tener en cuenta cuantas valoraciones tiene cada pelicula. Si una pelicula tiene muy pocas valoraciones, su media puede ser poco estable. Por eso se a?ade el parametro `min_ratings`, que prioriza recomendaciones similares con un volumen minimo de opiniones.

Esta mejora no sustituye la similitud coseno, sino que la complementa. El resultado sigue mostrando `similarity_score`, `rating_mean` y `rating_count` para que la interpretacion sea sencilla.

## 9. Funcion de recomendacion

La funcion `recommend_movies_by_genres` recibe el dataset, una matriz de similitud o la matriz de caracteristicas, un titulo, el numero de recomendaciones y un minimo de valoraciones. En este caso usamos la matriz de caracteristicas para que el calculo sea mas ligero con el dataset completo.

In [44]:
recommendations_toy_story = recommend_movies_by_genres(
    movies_clean,
    feature_matrix,
    'Toy Story',
    n=5,
)
recommendations_toy_story

Pelicula seleccionada: Toy Story (1995)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Missing Link,2019.0,Adventure|Animation|Children|Comedy|Fantasy,3.011952,251,1.0
1,DuckTales: The Movie - Treasure of the Lost Lamp,1990.0,Adventure|Animation|Children|Comedy|Fantasy,3.274590,244,1.0
2,The SpongeBob Movie: Sponge on the Run,2020.0,Adventure|Animation|Children|Comedy|Fantasy,2.706897,87,1.0
3,Wonder Park,2019.0,Adventure|Animation|Children|Comedy|Fantasy,2.618644,59,1.0
4,Trolls Holiday,2017.0,Adventure|Animation|Children|Comedy|Fantasy,2.772727,33,1.0


## 10. Efecto del filtro

Para ver la diferencia, podemos comparar el mismo titulo con y sin restriccion de valoraciones.

In [46]:
print('Sin filtro de valoraciones')
display(recommend_movies_by_genres(movies_clean, feature_matrix, 'Toy Story', n=5))

print('Con minimo de 20 valoraciones')
display(recommend_movies_by_genres(movies_clean, feature_matrix, 'Toy Story'))

Sin filtro de valoraciones
Pelicula seleccionada: Toy Story (1995)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Missing Link,2019.0,Adventure|Animation|Children|Comedy|Fantasy,3.011952,251,1.0
1,DuckTales: The Movie - Treasure of the Lost Lamp,1990.0,Adventure|Animation|Children|Comedy|Fantasy,3.274590,244,1.0
2,The SpongeBob Movie: Sponge on the Run,2020.0,Adventure|Animation|Children|Comedy|Fantasy,2.706897,87,1.0
3,Wonder Park,2019.0,Adventure|Animation|Children|Comedy|Fantasy,2.618644,59,1.0
4,Trolls Holiday,2017.0,Adventure|Animation|Children|Comedy|Fantasy,2.772727,33,1.0


Con minimo de 20 valoraciones
Pelicula seleccionada: Toy Story (1995)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Soul,2020.0,Adventure|Animation|Children|Comedy|Fantasy,3.911948,4486,1.0
1,Frozen II,2019.0,Adventure|Animation|Children|Comedy|Fantasy,3.238904,1442,1.0
2,"Wild, The",2006.0,Adventure|Animation|Children|Comedy|Fantasy,2.518564,404,1.0
3,Missing Link,2019.0,Adventure|Animation|Children|Comedy|Fantasy,3.011952,251,1.0
4,DuckTales: The Movie - Treasure of the Lost Lamp,1990.0,Adventure|Animation|Children|Comedy|Fantasy,3.274590,244,1.0
5,The SpongeBob Movie: Sponge on the Run,2020.0,Adventure|Animation|Children|Comedy|Fantasy,2.706897,87,1.0
6,Wonder Park,2019.0,Adventure|Animation|Children|Comedy|Fantasy,2.618644,59,1.0
7,UglyDolls,2019.0,Adventure|Animation|Children|Comedy|Fantasy,2.194444,36,1.0
8,Trolls Holiday,2017.0,Adventure|Animation|Children|Comedy|Fantasy,2.772727,33,1.0
9,Riverdance: The Animated Adventure,2021.0,Adventure|Animation|Children|Comedy|Fantasy,1.750000,2,1.0


## 11. Pruebas con varias peliculas

Probamos la funcion con varios titulos para comprobar que encuentra coincidencias exactas, coincidencias parciales y tambien el caso en que no existe la pelicula buscada.

In [48]:
recommend_movies_by_genres(movies_clean, feature_matrix, 'Interstellar')

Pelicula seleccionada: Interstellar (2014)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Edge of Tomorrow,2014.0,Action|Sci-Fi|IMAX,3.958198,19760,0.816497
1,Gravity,2013.0,Action|Sci-Fi|IMAX,3.597484,18126,0.816497
2,Cloud Atlas,2012.0,Drama|Sci-Fi|IMAX,3.560508,8065,0.816497
3,The Amazing Spider-Man 2,2014.0,Action|Sci-Fi|IMAX,3.012546,4304,0.816497
4,Contagion,2011.0,Sci-Fi|Thriller|IMAX,3.504482,4239,0.816497
5,Monsters vs. Aliens,2009.0,Animation|Sci-Fi|IMAX,3.074721,3406,0.816497
6,Transcendence,2014.0,Drama|Sci-Fi|IMAX,3.222408,3298,0.816497
7,"Invasion, U.S.A.",1952.0,Sci-Fi,1.800000,10,0.707107
8,Raumpatrouille Orion - Rücksturz ins Kino,2003.0,Sci-Fi,2.000000,1,0.707107
9,"The Twentieth Century Tramp; or, Happy Hooliga...",1902.0,Sci-Fi,2.500000,1,0.707107


In [49]:
recommend_movies_by_genres(movies_clean, feature_matrix, 'The lobster', n=5)

Pelicula seleccionada: The Lobster (2015)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Future '38,2017.0,Comedy|Romance|Sci-Fi,3.600000,10,1.0
1,Andover,2018.0,Comedy|Romance|Sci-Fi,2.000000,9,1.0
2,Codependent Lesbian Space Alien Seeks Same,2011.0,Comedy|Romance|Sci-Fi,2.722222,9,1.0
3,(UN)Ideal Man,2020.0,Comedy|Romance|Sci-Fi,2.750000,6,1.0
4,The Pod Generation,2023.0,Comedy|Romance|Sci-Fi,2.800000,5,1.0


In [ ]:
recommend_movies_by_genres(movies_clean, feature_matrix, 'pelicula inventada', n=5)

## 12. Conclusiones

Este primer recomendador funciona bien como punto de partida porque utiliza una representacion muy simple y facil de interpretar. Si dos peliculas comparten generos, el sistema es capaz de detectarlo y proponer recomendaciones coherentes de forma rapida.

La mejora con `min_ratings` ayuda a evitar que aparezcan en primeras posiciones peliculas con muy pocas valoraciones. Aun asi, la limitacion principal sigue siendo que solo usamos generos. Muchas peliculas pueden parecer iguales aunque tengan estilos, temas o tonos diferentes, por lo que sigue siendo una base util pero todavia limitada para un sistema mas completo.